# Transform Orders Data - String to JSON
1. Pre-process the JSON String to fix the Data Quality issues
2. Transform JSON String to JSON Object
3. Write transformed data to the silver schema

In [0]:
df = spark.table("gizmobox.bronze.py_orders")
display(df)

## 1. Pre-process the JSON String to fix the Data Quality issues

In [0]:
from pyspark.sql import functions as F

df_fixed_orders = (
    df.select(
        F.regexp_replace(F.col('value'), '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias("fixed_value")
    )
)
display(df_fixed_orders)

## 2. Transform JSON String to JSON Object

In [0]:
df_with_schema = (
    df_fixed_orders.select(
        F.schema_of_json (F.col('fixed_value')).alias('schema')
    )
)
display(df_with_schema.limit(1))

In [0]:
orders_schema = """
    STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>
"""

In [0]:
df_json_orders = (
    df_fixed_orders.select(
        F.from_json("fixed_value", orders_schema).alias("json_value")
    )
)

display(df_json_orders)

## 3. Write transformed data to the silver schema

In [0]:
df_json_orders.writeTo("gizmobox.silver.py_orders_json").createOrReplace()

In [0]:
df = spark.table("gizmobox.silver.py_orders_json")
display(df)